# 04. 평가 · 오류 분석

|  |  |
|---|---|
| **입력** | `runs/<실험>/weights/best.pt`, `data/pill_yolo/val_gt.json`, `outputs/image_conditions.csv`, `outputs/class_meta.csv` |
| **출력** | `experiments/results.csv`, `outputs/figures/`, `outputs/predictions/` |

---

## 무엇을 보는가

1. **mAP** — 대회 지표
2. **오류 유형 분해** — 어떤 처방이 필요한지
3. **촬영 조건별 성능** — `light_color`, `back_color`, `camera_la`, `drug_dir` 별 분해
4. **EDA 가설 검증** — (모양, 색)이 겹치는 약이 실제로 헷갈리는가
5. **★ Copy&Paste 효과 검증** — 크롭을 보유한 클래스가 실제로 더 잘 나오는가
6. **confidence sweep** — 운영 임계값 → 비즈니스 시사점

3번과 5번이 이 프로젝트의 차별점입니다.
사용자는 다양한 조명·배경·각도에서 촬영합니다.
"어두운 조명에서 mAP 가 0.12 떨어진다" 는 결과는 곧 제품 개선 방향이 됩니다.

## ★ 이전 버전에서 고친 것

| 이전 | 지금 | 이유 |
|---|---|---|
| `pilldata/image_conditions.csv` 참조 | `outputs/image_conditions.csv` | **아무도 그 파일을 만들지 않아 조건 분석이 통째로 죽어 있었습니다.** 01 이 생성합니다 |
| `train_meta.csv` 를 약 **이름**으로 조인 | `class_meta.csv` 를 **`category_id`** 로 조인 | 이름은 중복·표기 흔들림이 있어 조인이 샜습니다 |
| 조건을 `image_id` 로 조회 | **`file_name`** 으로 조회 | `val_gt.json` 의 id 는 0..N-1 재부여 값이라 원본 인덱스와 다릅니다 |
| `hypothesis.json` 참조 | 제거 | 생성되지 않는 파일이었습니다 |
| `IMGSZ = 640` | 학습과 동일하게 맞춤 | 학습/평가 해상도가 다르면 점수가 왜곡됩니다 |

## 왜 로컬 평가인가
캐글 제출은 **팀 전체 하루 10회**입니다. 로컬 val mAP 로 걸러서
확신 있는 시도에만 쓰세요.

In [ ]:
# ═══════════════ 설정 ═══════════════
DATA_ROOT = r"D:/X"
#1. 팀 구글드라이브에 있는 PillData의 압축을 푼다. 
#2. 새로운 파일을 생성한다.
#3. 그 파일에 압축을 푼 파일을 넣고, 새로운 파일의 경로주소를 적는다.
# ex)D드라이브 안에 있는 X라는 이름의 파일에 PillData파일을 넣었다. 그럼 D:/X 로 설정

EXP_NAME  = "exp_offline"                    # ★ 03 의 EXP_NAME 과 동일하게

WEIGHTS   = f"{DATA_ROOT}/runs/{EXP_NAME}/weights/best.pt"
DATA_DIR  = f"{DATA_ROOT}/data/pill_yolo"    # 02 산출물 (기준선이면 pill_yolo_base)
GT_JSON   = f"{DATA_DIR}/val_gt.json"
VAL_IMG   = f"{DATA_DIR}/images/val"
CMAP_JSON = f"{DATA_DIR}/category_map.json"

OUT_ROOT  = f"{DATA_ROOT}/outputs"
COND_CSV  = f"{OUT_ROOT}/image_conditions.csv"   # ★ 01 이 생성
META_CSV  = f"{OUT_ROOT}/class_meta.csv"         # ★ 01 이 생성

IMGSZ     = 960        # ★ 03 의 IMGSZ 와 반드시 동일하게
CONF      = 0.001      # ★ mAP 용이므로 매우 낮게
IOU_NMS   = 0.7
MAX_DET   = 100
TOPK      = 0          # 이미지당 상위 N개 (0=전부)
EVAL_CONF = 0.25       # 오류 분해용 임계값

PRED_CSV  = f"{OUT_ROOT}/predictions/{EXP_NAME}_val.csv"
FIG_DIR   = f"{OUT_ROOT}/figures"
RESULTS   = f"{DATA_ROOT}/experiments/results.csv"
LOG_JSONL = f"{DATA_ROOT}/experiments/log.jsonl"

SEED = 42

import os
for d in (os.path.dirname(PRED_CSV), FIG_DIR, f"{DATA_ROOT}/experiments"):
    os.makedirs(d, exist_ok=True)
print(f"평가 대상: {EXP_NAME}")
print(f"가중치   : {WEIGHTS}  존재 {os.path.exists(WEIGHTS)}")
print(f"평가 해상도 {IMGSZ} — 03 의 IMGSZ 와 같은지 확인하세요")

In [ ]:
"""공통 유틸 — 이 셀을 먼저 실행하세요."""
import os, json, glob, csv, random
from collections import Counter, defaultdict

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont

random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass
print(f"SEED = {SEED} (random / numpy / torch 고정)")

# ─────────────────────────────────────────────────────────────────────
# AI Hub 경구약제 데이터 명세 — 이 노트북이 쓰는 필드
#
#  [A] 촬영 조건 (이미지마다 다름)  → outputs/image_conditions.csv
#      light_color 촬영조명   back_color 촬영배경
#      camera_la   카메라위도  drug_dir   알약방향(앞/뒤)   drug_S 알약상태
#
#  [B] 약품 외형 (약마다 고정)      → outputs/class_meta.csv
#      drug_shape 모양   color_class1/2 색상   print_front 각인
#      leng_long/leng_short 크기   weight 증강가중치   n_crops 보유 크롭 수
# ─────────────────────────────────────────────────────────────────────
COND_KEYS = ["light_color", "back_color", "camera_la", "drug_dir", "drug_S"]


def imread_unicode(path):
    try:
        return cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        return None


def imwrite_unicode(path, img):
    ext = os.path.splitext(str(path))[1] or ".png"
    ok, buf = cv2.imencode(ext, img)
    if not ok:
        return False
    buf.tofile(str(path)); return True


def find_korean_font():
    cands = ["C:/Windows/Fonts/malgun.ttf",
             "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
             "/System/Library/Fonts/AppleSDGothicNeo.ttc"]
    for pat in ("/usr/share/fonts/**/*CJK*.ttc", "/usr/share/fonts/**/*Nanum*.ttf",
                "/usr/share/fonts/**/*Gothic*.ttf"):
        cands += sorted(glob.glob(pat, recursive=True))
    for p in cands:
        if os.path.exists(p):
            try:
                ImageFont.truetype(p, 20); return p
            except Exception:
                pass
    return None

FONT_PATH = find_korean_font()
print("한글 폰트:", FONT_PATH or "⚠️ 미발견")

_PALETTE = [(255,89,94),(56,176,0),(25,130,196),(255,202,58),(138,80,220),
            (0,187,249),(241,91,181),(155,200,60),(255,140,0),(0,200,170),
            (200,60,120),(120,160,255)]

def class_color(cid):
    return _PALETTE[int(cid) % len(_PALETTE)]


def draw_detections(img_bgr, dets, id2name=None, font_scale=1.0,
                    box_thickness=3, show_conf=True):
    img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    H, W = img_bgr.shape[:2]
    size = max(16, int(min(W, H) * 0.028 * font_scale))
    font = ImageFont.truetype(FONT_PATH, size) if FONT_PATH else ImageFont.load_default()
    for d in dets:
        cid = d["category_id"]
        x, y, w, h = [float(v) for v in d["bbox"]]
        color = class_color(cid)
        name = str(id2name.get(cid, cid)) if id2name else str(cid)
        label = f"{name} {d['score']:.2f}" if (show_conf and d.get("score") is not None) else name
        draw.rectangle([x, y, x + w, y + h], outline=color, width=box_thickness)
        tb = draw.textbbox((0, 0), label, font=font)
        tw, th = tb[2] - tb[0], tb[3] - tb[1]
        pad = max(3, size // 6)
        ly = y - th - pad * 2
        if ly < 0:
            ly = y + pad
        lx = min(max(0, x), W - tw - pad * 2)
        draw.rectangle([lx, ly, lx + tw + pad*2, ly + th + pad*2], fill=color)
        lum = 0.299*color[0] + 0.587*color[1] + 0.114*color[2]
        draw.text((lx+pad, ly+pad), label, font=font,
                  fill=(0,0,0) if lum > 150 else (255,255,255))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def save_per_image(records, img_dir, out_dir, id2name=None, conf_thr=0.0,
                   limit=0, suffix="", verbose=True):
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for fn, dets in sorted(records.items()):
        p = os.path.join(img_dir, fn)
        if not os.path.exists(p):
            p = os.path.join(img_dir, os.path.basename(fn))
            if not os.path.exists(p):
                continue
        img = imread_unicode(p)
        if img is None:
            continue
        keep = [d for d in dets if d.get("score") is None or d["score"] >= conf_thr]
        vis = draw_detections(img, keep, id2name)
        stem, ext = os.path.splitext(os.path.basename(fn))
        imwrite_unicode(os.path.join(out_dir, f"{stem}{suffix}{ext or '.png'}"), vis)
        n += 1
        if limit and n >= limit:
            break
    if verbose:
        print(f"{n}장 저장 → {out_dir}  (이미지 1장 = 파일 1개)")
    return n


def show_image(path, max_width=900):
    from IPython.display import display
    im = Image.open(path)
    if im.width > max_width:
        im = im.resize((max_width, int(im.height * max_width / im.width)))
    display(im)


def load_coco(path):
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    by_img = defaultdict(list)
    for a in d.get("annotations", []):
        by_img[a["image_id"]].append(a)
    return d, by_img


def setup_matplotlib():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    if FONT_PATH:
        try:
            font_manager.fontManager.addfont(FONT_PATH)
            plt.rcParams["font.family"] = font_manager.FontProperties(fname=FONT_PATH).get_name()
        except Exception:
            pass
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 110
    return plt

print("공통 유틸 로드 완료")

---
## 1. val 예측 생성

**★ `CONF=0.001` 이 핵심입니다.**
mAP 는 precision-recall 곡선 아래 면적이라 **낮은 confidence 예측도 recall 을 올려
점수에 기여**합니다. `conf=0.5` 로 자르는 건 가장 흔한 실수입니다.

> `images/val` 의 이미지는 02 에서 **전처리가 적용된 상태**로 저장되어 있습니다.
> 학습 이미지와 동일한 상태이므로 여기서 추가 전처리를 걸면 안 됩니다 (이중 적용).

In [ ]:
from ultralytics import YOLO

CMAP = json.load(open(CMAP_JSON, encoding="utf-8"))
IDX2CAT = {int(k): int(v) for k, v in CMAP["idx2cat"].items()}
IDXNAME = {int(k): v for k, v in CMAP["names"].items()}
# ★ names 는 YOLO 인덱스 키입니다. category_id 로 다시 매핑해야 합니다.
ID2NAME = {IDX2CAT[i]: n for i, n in IDXNAME.items()}

gt_coco, gt_by_img = load_coco(GT_JSON)
file2id = {os.path.basename(im["file_name"]): im["id"] for im in gt_coco["images"]}
id2file = {im["id"]: os.path.basename(im["file_name"]) for im in gt_coco["images"]}

model = YOLO(WEIGHTS)
paths = sorted(glob.glob(f"{VAL_IMG}/*"))
print(f"val 이미지 {len(paths):,}장 / GT 이미지 {len(gt_coco['images']):,}장")
_miss = [p for p in paths if os.path.basename(p) not in file2id]
if _miss:
    print(f"⚠️ GT 에 없는 이미지 {len(_miss)}장 — val_gt.json 과 images/val 이 어긋났습니다")

rows, ann_id = [], 1
for i in range(0, len(paths), 8):
    chunk = paths[i:i+8]
    for p, r in zip(chunk, model.predict(chunk, conf=CONF, iou=IOU_NMS, imgsz=IMGSZ,
                                         max_det=MAX_DET, verbose=False)):
        iid = file2id.get(os.path.basename(p))
        if iid is None or r.boxes is None or len(r.boxes) == 0:
            continue
        xyxy = r.boxes.xyxy.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        cls  = r.boxes.cls.cpu().numpy().astype(int)
        order = conf.argsort()[::-1]
        if TOPK:
            order = order[:TOPK]
        for j in order:
            x1, y1, x2, y2 = xyxy[j]
            rows.append([ann_id, iid, IDX2CAT[int(cls[j])],
                         round(float(x1), 2), round(float(y1), 2),
                         round(float(x2-x1), 2), round(float(y2-y1), 2),
                         round(float(conf[j]), 5)])
            ann_id += 1

with open(PRED_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["annotation_id","image_id","category_id",
                "bbox_x","bbox_y","bbox_w","bbox_h","score"])
    w.writerows(rows)
print(f"검출 {len(rows):,}개 → {PRED_CSV}")

---
## 2. 평가 함수

COCO 101-point 보간으로 AP 를 계산합니다.
`only_images` 인자를 주면 **그 이미지들만** 평가하므로,
같은 함수로 촬영 조건별 분해까지 할 수 있습니다.

In [ ]:
def iou_xywh(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ax1, ay1 = a[:,0][:,None], a[:,1][:,None]
    ax2, ay2 = ax1 + a[:,2][:,None], ay1 + a[:,3][:,None]
    bx1, by1 = b[:,0][None,:], b[:,1][None,:]
    bx2, by2 = bx1 + b[:,2][None,:], by1 + b[:,3][None,:]
    iw = np.clip(np.minimum(ax2,bx2) - np.maximum(ax1,bx1), 0, None)
    ih = np.clip(np.minimum(ay2,by2) - np.maximum(ay1,by1), 0, None)
    inter = iw * ih
    union = (a[:,2]*a[:,3])[:,None] + (b[:,2]*b[:,3])[None,:] - inter
    return np.where(union > 0, inter/np.maximum(union,1e-9), 0.0)


def average_precision(tp, conf, n_gt):
    """COCO 101-point 보간."""
    if n_gt == 0:
        return float("nan")
    if len(tp) == 0:
        return 0.0
    o = np.argsort(-conf); tp = tp[o]
    ctp, cfp = np.cumsum(tp), np.cumsum(1-tp)
    rec, prec = ctp/n_gt, ctp/np.maximum(ctp+cfp, 1e-9)
    for i in range(len(prec)-2, -1, -1):
        prec[i] = max(prec[i], prec[i+1])
    q = np.linspace(0,1,101); idx = np.searchsorted(rec, q, side="left")
    p = np.zeros_like(q); v = idx < len(prec); p[v] = prec[idx[v]]
    return float(p.mean())


def load_pred_csv(path):
    preds = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            preds[(int(r["image_id"]), int(r["category_id"]))].append(
                np.array([float(r["bbox_x"]), float(r["bbox_y"]),
                          float(r["bbox_w"]), float(r["bbox_h"]),
                          float(r["score"])]))
    return {k: np.array(v) for k, v in preds.items()}


def load_gt(path):
    coco, _ = load_coco(path)
    gts = defaultdict(list)
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        gts[(int(a["image_id"]), int(a["category_id"]))].append(
            np.array([x, y, w, h]))
    return {k: np.array(v) for k, v in gts.items()}


def evaluate(preds, gts, iou_thrs, only_images=None):
    """only_images 를 주면 그 이미지들만 평가 (조건별 분해에 사용)."""
    if only_images is not None:
        S = set(only_images)
        preds = {k: v for k, v in preds.items() if k[0] in S}
        gts   = {k: v for k, v in gts.items()   if k[0] in S}
    cats = sorted({c for _, c in gts} | {c for _, c in preds})
    imgs = sorted({i for i, _ in gts} | {i for i, _ in preds})
    ap = {t: {} for t in iou_thrs}
    for cat in cats:
        n_gt = sum(len(gts.get((i,cat),[])) for i in imgs)
        rr = [(i,p) for i in imgs for p in preds.get((i,cat),[])]
        if not rr:
            for t in iou_thrs:
                ap[t][cat] = 0.0 if n_gt else float("nan")
            continue
        conf = np.array([r[1][4] for r in rr]); o = np.argsort(-conf)
        rr = [rr[k] for k in o]; conf = conf[o]
        for t in iou_thrs:
            matched = defaultdict(set); tp = np.zeros(len(rr))
            for k,(img,box) in enumerate(rr):
                g = gts.get((img,cat))
                if g is None or len(g)==0:
                    continue
                ious = iou_xywh(box[None,:4], g[:,:4])[0]
                cand = [(v,j) for j,v in enumerate(ious) if v>=t and j not in matched[img]]
                if cand:
                    _, j = max(cand); matched[img].add(j); tp[k] = 1
            ap[t][cat] = average_precision(tp, conf, n_gt)
    out = {}
    for t in iou_thrs:
        v = [x for x in ap[t].values() if not np.isnan(x)]
        out[f"mAP@{t:.2f}"] = float(np.mean(v)) if v else 0.0
    out["mAP@[.5:.95]"] = float(np.mean([out[f"mAP@{t:.2f}"] for t in iou_thrs]))
    out["_per_cat"] = ap[iou_thrs[0]]
    return out


def error_breakdown(preds, gts, conf_thr=0.25, iou_thr=0.5, only_images=None):
    if only_images is not None:
        S = set(only_images)
        preds = {k: v for k, v in preds.items() if k[0] in S}
        gts   = {k: v for k, v in gts.items()   if k[0] in S}
    gt_by, pr_by = defaultdict(list), defaultdict(list)
    for (i,cc), arr in gts.items():
        for b in arr:
            gt_by[i].append((cc, np.array(b[:4])))
    for (i,cc), arr in preds.items():
        for b in arr:
            if b[4] >= conf_thr:
                pr_by[i].append((cc, b[:4], b[4]))

    tp = mis = loc = bg = matched_total = 0
    n_gt = sum(len(v) for v in gt_by.values())
    confusion = Counter()
    for i in sorted(set(gt_by) | set(pr_by)):
        gl = gt_by.get(i, []); pl = sorted(pr_by.get(i, []), key=lambda x: -x[2])
        used = set()
        for cc, box, _ in pl:
            if not gl:
                bg += 1; continue
            gb = np.array([g[1] for g in gl])
            ious = iou_xywh(np.array(box)[None,:], gb)[0]
            j = int(np.argmax(ious))
            if ious[j] < 0.1:      bg += 1
            elif ious[j] < iou_thr: loc += 1
            elif gl[j][0] != cc:   mis += 1; confusion[(gl[j][0], cc)] += 1
            elif j in used:        bg += 1
            else:                  tp += 1; used.add(j)
        matched_total += len(used)
    fn = n_gt - matched_total
    tot = mis + loc + bg + fn
    return {"n_gt": n_gt, "TP": tp, "미검출(FN)": fn, "오분류": mis,
            "위치오차": loc, "배경오검출": bg,
            "recall": tp/max(1,n_gt), "precision": tp/max(1,tp+mis+loc+bg),
            "ratio": {"미검출(FN)": fn/max(1,tot), "오분류": mis/max(1,tot),
                      "위치오차": loc/max(1,tot), "배경오검출": bg/max(1,tot)},
            "confusions": confusion.most_common(20)}

In [ ]:
preds, gts = load_pred_csv(PRED_CSV), load_gt(GT_JSON)
IOU_THRS = np.arange(0.5, 0.96, 0.05)
res = evaluate(preds, gts, IOU_THRS)
eb  = error_breakdown(preds, gts, conf_thr=EVAL_CONF)

print(f"mAP@0.5        {res['mAP@0.50']:.4f}   ← 대회 지표일 가능성 높음")
print(f"mAP@0.75       {res['mAP@0.75']:.4f}")
print(f"mAP@[.5:.95]   {res['mAP@[.5:.95]']:.4f}")
print(f"\nrecall {eb['recall']:.4f}  precision {eb['precision']:.4f}  (conf≥{EVAL_CONF})")

---
## 3. 오류 유형 분해

`mAP 0.52` 만 쓰면 숫자지만, **"오류의 63%가 오분류"** 는 인사이트입니다.
두 결론은 완전히 다른 처방으로 이어집니다.

| 오류 유형 | 뜻 | 처방 |
|---|---|---|
| 미검출(FN) | 알약이 있는데 못 찾음 | 재현율 문제 → `IMGSZ` ↑, `CONF` ↓, Copy&Paste 배치 다양성 ↑ |
| 오분류 | 박스는 맞는데 이름이 틀림 | **각인 판독 문제** → `IMGSZ` ↑, 2-stage |
| 위치오차 | 박스가 부정확 | `mosaic`/`scale` 재검토 |
| 배경오검출 | 없는 걸 찾음 | `CONF` ↑, NMS iou 조정 |

In [ ]:
plt = setup_matplotlib()

for k, v in sorted(eb["ratio"].items(), key=lambda x: -x[1]):
    print(f"  {k:<12} {eb[k]:>6,}개 ({v*100:5.1f}%) {'█'*int(v*40)}")

top = max(eb["ratio"], key=eb["ratio"].get)
advice = {"미검출(FN)": "재현율 문제. IMGSZ 를 키우거나 CONF 를 낮추세요. "
                        "Copy&Paste 로 배치 다양성을 늘리는 것도 유효합니다.",
          "오분류": "검출은 되는데 종류를 틀립니다. 해상도를 올려 각인 판독력을 높이세요.",
          "위치오차": "박스가 부정확합니다. IMGSZ 상향, mosaic/scale 재검토.",
          "배경오검출": "없는 걸 찾습니다. CONF 상향 또는 NMS iou 조정."}
print(f"\n★ 최대 오류: {top}\n  처방: {advice[top]}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
labels = list(eb["ratio"]); vals = [eb["ratio"][k] for k in labels]
axes[0].pie(vals, labels=labels, autopct="%1.1f%%",
            colors=["#ff6b6b","#ffd93d","#6bcB77","#4d96ff"])
axes[0].set_title("오류 유형 구성")
axes[1].bar(["TP"]+labels, [eb["TP"]]+[eb[k] for k in labels],
            color=["#06d6a0","#ff6b6b","#ffd93d","#6bcB77","#4d96ff"])
axes[1].set_ylabel("개수"); axes[1].set_title("검출 결과 분해")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{EXP_NAME}_error_breakdown.png", bbox_inches="tight")
plt.show()

---
## 4. 촬영 조건별 성능 — 이 프로젝트의 차별점

사용자는 다양한 조명·배경·각도에서 촬영합니다.
`light_color`, `back_color`, `camera_la`, `drug_dir` 별로 mAP 를 분해하면
**어떤 상황에서 제품이 실패하는지** 정량적으로 드러납니다.

> "전구색 조명에서 mAP 가 0.12 낮다" → 앱에 촬영 가이드를 넣거나
> 해당 조건 데이터를 보강해야 한다는 **구체적 개선 방향**이 나옵니다.

### ★ 조인 키에 주의

`val_gt.json` 의 `image_id` 는 val 안에서 **0부터 다시 매긴 번호**입니다.
01 의 `image_conditions.csv` 는 **전체 데이터셋 인덱스**를 씁니다.
둘은 다른 값이므로 **`file_name` 으로 조인**해야 합니다.
(이전 버전은 `image_id` 로 조인해 조건이 전부 어긋나 있었습니다.)

In [ ]:
# ★ file_name 으로 조인합니다 (image_id 는 split 마다 재부여되므로 쓰면 안 됩니다)
cond_by_file = {}
if os.path.exists(COND_CSV):
    with open(COND_CSV, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            cond_by_file[os.path.basename(r["file_name"])] = r
    print(f"촬영 조건 로드: {len(cond_by_file):,}장 ({COND_CSV})")
else:
    print(f"⚠️  {COND_CSV} 가 없습니다. 01_eda 를 끝까지 실행하세요.")

val_ids = [im["id"] for im in gt_coco["images"]]
matched = sum(1 for i in val_ids if id2file.get(i) in cond_by_file)
print(f"조인 성공 {matched}/{len(val_ids)}장",
      "✅" if matched == len(val_ids) else "⚠️ 파일명이 어긋납니다")

cond_results = {}
for key in COND_KEYS:
    groups = defaultdict(list)
    for iid in val_ids:
        row = cond_by_file.get(id2file.get(iid, ""), {})
        v = str(row.get(key, "")).strip()
        groups[v if v else "미상"].append(iid)
    rows_ = []
    for v, ids in sorted(groups.items()):
        if len(ids) < 3:                 # 표본이 너무 적으면 신뢰할 수 없음
            continue
        r = evaluate(preds, gts, np.array([0.5]), only_images=ids)
        e = error_breakdown(preds, gts, conf_thr=EVAL_CONF, only_images=ids)
        rows_.append((v, len(ids), r["mAP@0.50"], e["recall"], e["precision"]))
    if len(rows_) >= 2:
        cond_results[key] = rows_

if not cond_results:
    print("\n조건별 비교 불가 — val 표본이 적거나 조건이 단일값입니다.")
    print("(val 20장 내외면 조건마다 3장 미만이 되기 쉽습니다. "
          "test 까지 합쳐 보거나 분할 비율을 조정하세요.)")

for key, rows_ in cond_results.items():
    print(f"\n■ {key}")
    print(f"   {'값':<16}{'이미지':>7}{'mAP@0.5':>10}{'recall':>9}{'precision':>11}")
    for v, n, m, rc, pr in sorted(rows_, key=lambda x: -x[2]):
        print(f"   {str(v)[:15]:<16}{n:>7}{m:>10.4f}{rc:>9.4f}{pr:>11.4f}")
    best, worst = max(rows_, key=lambda x: x[2]), min(rows_, key=lambda x: x[2])
    gap = best[2] - worst[2]
    if gap > 0.03:
        print(f"   ★ 격차 {gap:.4f}  ('{best[0]}' 최고 / '{worst[0]}' 최저)")
        print(f"      → 이 조건이 성능에 유의미한 영향을 줍니다. 보고서에 인용하세요.")

In [ ]:
if cond_results:
    n = len(cond_results)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4.2))
    if n == 1:
        axes = [axes]
    for ax, (key, rows_) in zip(axes, cond_results.items()):
        rows_ = sorted(rows_, key=lambda x: -x[2])
        ax.bar([str(v)[:10] for v,_,_,_,_ in rows_], [m for _,_,m,_,_ in rows_],
               color="#3a86ff")
        for i, (v,cnt,m,_,_) in enumerate(rows_):
            ax.text(i, m, f"{m:.3f}\n(n={cnt})", ha="center", va="bottom", fontsize=8)
        ax.set_title(key); ax.set_ylabel("mAP@0.5")
        ax.tick_params(axis="x", rotation=25); ax.grid(alpha=0.3, axis="y")
    plt.suptitle("촬영 조건별 성능", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{EXP_NAME}_condition_breakdown.png", bbox_inches="tight")
    plt.show()
else:
    print("그래프 생략 (조건별 비교 불가)")

---
## 5. EDA 가설 검증 + Copy&Paste 효과

### 가설 1 (01 의 EDA)
> 같은 `(모양, 색상1)` 을 공유하는 약이 오분류 상위에 오를 것이다.

맞으면 → "모델의 한계는 검출이 아니라 **외형이 동일한 약의 각인 판독**" 이라는
명확한 결론이 나옵니다. 처방은 해상도 상향이지 증강이 아닙니다.

### 가설 2 (02 의 Copy&Paste)
> `cropped_pills_review` 에 크롭이 있어 합성에 많이 등장한 클래스가 더 잘 나올 것이다.

`class_meta.csv` 의 `n_crops` 로 클래스를 갈라 AP 를 비교합니다.

> ⚠️ 이건 **인과가 아니라 상관**입니다. 크롭 보유 여부는 클래스 난이도와도
> 얽혀 있습니다. 인과를 보려면 03 에서 `exp_cp0` (`N_SYNTH=0`) 와 비교하세요.

### ★ 조인 키
`class_meta.csv` 를 **`category_id`** 로 조인합니다.
이전 버전은 약 **이름**으로 조인해 표기가 조금만 달라도 조인이 샜습니다.

In [ ]:
# ─────────── 클래스 메타 로드 (category_id 로 조인) ───────────
CLS_META = {}
if os.path.exists(META_CSV):
    with open(META_CSV, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            CLS_META[int(r["category_id"])] = r
    print(f"클래스 메타 로드: {len(CLS_META)}종 ({META_CSV})")
else:
    print(f"⚠️  {META_CSV} 가 없습니다. 01_eda 를 끝까지 실행하세요.")


def m_get(cid, key, default=""):
    return (CLS_META.get(int(cid), {}) or {}).get(key, default)


def to_int(v, d=0):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return d


# ═══════════ 가설 1: 오분류 쌍의 외형 동일 비율 ═══════════
print("\n■ 자주 헷갈리는 쌍 (정답 → 예측)")
same_pair = tot_pair = 0
for (g, p), n in eb["confusions"][:12]:
    gn, pn = ID2NAME.get(g, str(g)), ID2NAME.get(p, str(p))
    tag = ""
    if g in CLS_META and p in CLS_META:
        tot_pair += 1
        sa = (m_get(g, "drug_shape"), m_get(g, "color_class1"))
        sb = (m_get(p, "drug_shape"), m_get(p, "color_class1"))
        if sa == sb:
            same_pair += 1; tag = f"  ★ 외형 동일 {sa}"
        else:
            tag = f"  {sa} vs {sb}"
        try:
            d = abs(float(m_get(g, "leng_long") or 0) - float(m_get(p, "leng_long") or 0))
            tag += f" | 장축차 {d:.1f}mm"
        except ValueError:
            pass
    print(f"  {gn[:20]:<22} → {pn[:20]:<22} {n:>3}회{tag}")

if tot_pair:
    ratio = same_pair / tot_pair
    print(f"\n★ 오분류 상위 {tot_pair}쌍 중 {same_pair}쌍이 (모양,색상1) 동일 ({ratio*100:.0f}%)")
    if ratio >= 0.5:
        print("  → 가설 1 검증됨. 보고서에 다음과 같이 쓰세요:")
        print(f'     "오분류 상위 {tot_pair}쌍 중 {same_pair}쌍({ratio*100:.0f}%)이 동일한')
        print('      (모양, 색상) 조합이었다. 즉 모델의 한계는 검출이 아니라 외형이')
        print('      같은 약의 각인 판독에 있으며, 해상도 상향이 유효한 처방이다."')
    else:
        print("  → 부분적으로만 맞습니다. 크기·각인 등 다른 요인을 확인하세요.")
else:
    print("\n오분류 쌍이 없거나 메타 조인에 실패했습니다.")

# ═══════════ 가설 2: Copy&Paste 크롭 보유 여부별 AP ═══════════
per_cat = {c: v for c, v in res["_per_cat"].items() if not np.isnan(v)}
have = [ap for c, ap in per_cat.items() if to_int(m_get(c, "n_crops")) > 0]
none = [ap for c, ap in per_cat.items() if to_int(m_get(c, "n_crops")) == 0]

print(f"\n■ Copy&Paste 크롭 보유 여부별 AP@0.5")
print(f"{'그룹':<28}{'클래스수':>8}{'AP50':>9}")
print("-" * 46)
print(f"{'크롭 있음 (합성에 등장)':<28}{len(have):>8}"
      f"{np.mean(have) if have else float('nan'):>9.3f}")
print(f"{'크롭 없음':<28}{len(none):>8}"
      f"{np.mean(none) if none else float('nan'):>9.3f}")
if have and none:
    gap = np.mean(have) - np.mean(none)
    print(f"\n★ 격차 {gap:+.3f}")
    print("  (상관 지표입니다. 인과를 보려면 03 의 exp_cp0 (N_SYNTH=0) 와 비교하세요.)")

# 크롭 수 구간별
if CLS_META:
    print(f"\n■ 보유 크롭 수 구간별 AP@0.5")
    print(f"{'구간':<20}{'클래스수':>8}{'AP50':>9}")
    print("-" * 38)
    for lo, hi, nm in [(0, 0, "0장"), (1, 9, "1~9장"),
                       (10, 29, "10~29장"), (30, 10**9, "30장 이상")]:
        v = [ap for c, ap in per_cat.items()
             if lo <= to_int(m_get(c, "n_crops")) <= hi]
        if v:
            print(f"{nm:<20}{len(v):>8}{np.mean(v):>9.3f}")

# weight 구간별
w_of = {c: float(m_get(c, "weight") or 1.0) for c in per_cat}
if w_of and np.std(list(w_of.values())) > 1e-6:
    wq = np.percentile(list(w_of.values()), [25, 50, 75])
    print(f"\n■ 증강 가중치 4분위별 AP@0.5")
    print(f"{'구간':<20}{'클래스수':>8}{'AP50':>9}")
    print("-" * 38)
    for lo, hi, nm in [(-1e9, wq[0], "Q1 (쉬움)"), (wq[0], wq[1], "Q2"),
                       (wq[1], wq[2], "Q3"), (wq[2], 1e9, "Q4 (어려움)")]:
        v = [ap for c, ap in per_cat.items() if lo < w_of[c] <= hi]
        if v:
            print(f"{nm:<20}{len(v):>8}{np.mean(v):>9.3f}")
    print("\n★ Q4 가 Q1 대비 크게 낮으면 증강이 아직 부족한 것입니다.")
    print("  → N_SYNTH ↑ 또는 IMGSZ ↑ (각인 해상도가 병목이면 증강으론 안 됩니다)")

# 하위 10종
print(f"\n■ AP50 하위 10종")
print(f"{'클래스':<24}{'각인':<9}{'색상1':<8}{'w':>6}{'크롭':>6}{'AP50':>8}")
print("-" * 62)
for cid, ap in sorted(per_cat.items(), key=lambda kv: kv[1])[:10]:
    print(f"{str(ID2NAME.get(cid, cid))[:22]:<24}"
          f"{str(m_get(cid,'print_front') or '-')[:7]:<9}"
          f"{str(m_get(cid,'color_class1') or '-')[:6]:<8}"
          f"{float(m_get(cid,'weight') or 1):>6.2f}"
          f"{to_int(m_get(cid,'n_crops')):>6}{ap:>8.3f}")

### 5-1. 틀린 사례 — **이미지 1장 = 파일 1개**, 정답/예측 별도 저장

In [ ]:
ERR_DIR = f"{OUT_ROOT}/predictions/{EXP_NAME}_errors"

bad = []
for iid, fn in id2file.items():
    n_gt = sum(len(v) for (i,_), v in gts.items() if i == iid)
    n_pr = sum(1 for (i,_), arr in preds.items() if i == iid
               for b in arr if b[4] >= EVAL_CONF)
    if n_gt != n_pr:
        bad.append((abs(n_gt-n_pr), iid, fn))
bad.sort(reverse=True)
print(f"개수가 틀린 이미지 {len(bad)}장 / 전체 {len(id2file)}장 (상위 6장 저장)")

gt_rec, pr_rec = {}, {}
for _, iid, fn in bad[:6]:
    gt_rec[fn] = [{"category_id": cc, "bbox": list(b)}
                  for (i,cc), arr in gts.items() if i == iid for b in arr]
    pr_rec[fn] = [{"category_id": cc, "bbox": list(b[:4]), "score": float(b[4])}
                  for (i,cc), arr in preds.items() if i == iid
                  for b in arr if b[4] >= EVAL_CONF]

save_per_image(gt_rec, VAL_IMG, ERR_DIR, id2name=ID2NAME, suffix="_1정답")
save_per_image(pr_rec, VAL_IMG, ERR_DIR, id2name=ID2NAME, suffix="_2예측")
for p in sorted(glob.glob(f"{ERR_DIR}/*"))[:4]:
    print(os.path.basename(p)); show_image(p)

---
## 6. confidence sweep → 비즈니스 시사점

mAP 최대화는 `CONF=0.001` 을 요구하지만,
**실제 앱에서는 사용자에게 오답을 보여주는 비용이 훨씬 큽니다.**
헬스케어 도메인이므로 이 대비가 특히 설득력이 있습니다.

In [ ]:
sweep = []
for t in (0.001, 0.05, 0.1, 0.25, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    e = error_breakdown(preds, gts, conf_thr=t)
    n = sum(1 for arr in preds.values() for b in arr if b[4] >= t)
    sweep.append((t, e["recall"], e["precision"], n))

print(" 임계값   recall  precision   검출수")
for t, r, p, n in sweep:
    print(f"  {t:<7} {r:.4f}    {p:.4f}   {n:>7,}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ts = [s[0] for s in sweep]
ax.plot(ts, [s[1] for s in sweep], "o-", label="recall", color="#3a86ff")
ax.plot(ts, [s[2] for s in sweep], "s-", label="precision", color="#ff006e")
ax.set_xlabel("confidence 임계값"); ax.set_ylabel("값")
ax.set_title("운영 임계값 결정"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{EXP_NAME}_conf_sweep.png", bbox_inches="tight")
plt.show()

cand = [s for s in sweep if s[1] >= 0.6]
best = max(cand, key=lambda s: s[2]) if cand else sweep[0]
print(f"\n예시 해석:")
print(f"  conf=0.001 일 때 precision {sweep[0][2]:.2f} → 검출의 "
      f"{(1-sweep[0][2])*100:.0f}% 가 오답입니다.")
print(f"  recall 0.6 이상을 유지하며 precision 이 최대인 지점은 conf={best[0]} "
      f"(recall {best[1]:.2f}, precision {best[2]:.2f}).")
print(f"  헬스케어 도메인에서는 이 값을 운영 임계값으로 두고")
print(f"  그 미만은 '확인 필요' 로 처리하는 것이 타당합니다.")
print(f"\n  ※ 제출용 CSV 는 mAP 를 위해 CONF=0.001 을 그대로 쓰세요 (05 참고).")

---
## 7. 실험 비교표 (ablation)

`experiments/log.jsonl` (03 이 누적) 과 이 노트북의 평가 결과를 합쳐
`experiments/results.csv` 에 한 줄씩 쌓습니다.

**증강 기여도를 보려면** `exp_base` / `exp_online` / `exp_offline` 을
각각 03 에서 학습하고 여기서 `EXP_NAME` 만 바꿔 재실행하세요.

In [ ]:
# ---------- 03 의 로그에서 이 실험의 설정을 찾아옴 ----------
note, n_train, off_aug = "", "", ""
if os.path.exists(LOG_JSONL):
    for line in open(LOG_JSONL, encoding="utf-8"):
        try:
            d = json.loads(line)
        except json.JSONDecodeError:
            continue
        if d.get("name") == EXP_NAME:
            note = d.get("note", "")
            n_train = d.get("n_train_images", "")
            oa = d.get("offline_aug") or {}
            off_aug = (f"geom×{oa.get('geom_mult','?')}+cp{oa.get('n_synth','?')}"
                       if oa else "")

row = {"실험": EXP_NAME, "설명": note,
       "학습이미지": n_train, "오프라인증강": off_aug,
       "mAP50": round(res["mAP@0.50"], 4),
       "mAP50_95": round(res["mAP@[.5:.95]"], 4),
       "recall": round(eb["recall"], 4), "precision": round(eb["precision"], 4),
       "FN": eb["미검출(FN)"], "오분류": eb["오분류"],
       "위치오차": eb["위치오차"], "배경FP": eb["배경오검출"]}

# 같은 실험명이 이미 있으면 갱신, 없으면 추가
rows_ = []
if os.path.exists(RESULTS):
    with open(RESULTS, encoding="utf-8-sig") as f:
        rows_ = [r for r in csv.DictReader(f) if r.get("실험") != EXP_NAME]
rows_.append(row)

with open(RESULTS, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(row))
    w.writeheader(); w.writerows(rows_)
print(f"누적 저장: {RESULTS}  ({len(rows_)}개 실험)")

cols = list(row)
print("\n| " + " | ".join(cols) + " |")
print("|" + "|".join(["---"]*len(cols)) + "|")
for r in rows_:
    print("| " + " | ".join(str(r.get(cc, "")) for cc in cols) + " |")

if len(rows_) >= 2:
    fig, ax = plt.subplots(figsize=(max(6, len(rows_)*1.8), 4.2))
    x = np.arange(len(rows_)); wd = 0.35
    ax.bar(x-wd/2, [float(r["mAP50"]) for r in rows_], wd,
           label="mAP@0.5", color="#3a86ff")
    ax.bar(x+wd/2, [float(r["mAP50_95"]) for r in rows_], wd,
           label="mAP@[.5:.95]", color="#8338ec")
    ax.set_xticks(x); ax.set_xticklabels([r["실험"] for r in rows_], rotation=20)
    ax.set_ylabel("mAP"); ax.set_title("실험별 성능"); ax.legend(); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/ablation.png", bbox_inches="tight")
    plt.show()
else:
    print("\n(실험이 1개뿐입니다. 03 에서 exp_base / exp_online 도 학습하면 비교표가 됩니다)")

---
## 8. 요약 — 보고서에 그대로

| 항목 | 값 |
|---|---|
| mAP@0.5 / mAP@[.5:.95] | |
| recall / precision | |
| 최대 오류 유형 | |
| 성능 격차가 큰 촬영 조건 | |
| 오분류 쌍의 외형 동일 비율 | |
| Copy&Paste 크롭 보유군 AP 격차 | |
| 권장 운영 임계값 | |

### 다음 노트북
`05_inference.ipynb` — 테스트 추론 및 제출 파일 생성